In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import math
import torch
import torch.nn as nn


In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, num_heads):
    super(MultiHeadAttention, self).__init__()
    assert d_model % num_heads == 0

    self.d_model = d_model
    self.num_heads = num_heads
    self.head_dim = d_model // num_heads

    #Linear Transformatioin for Q, K, V and final output
    #Linear function takes first parameter as an input dimension and changes dimension to second parameter
    #to change Weights while training (backpropagation + update weights)
    self.q_linear = nn.Linear(d_model, d_model)
    self.k_linear = nn.Linear(d_model, d_model)
    self.v_linear = nn.Linear(d_model, d_model)

    self.out_linear = nn.Linear(d_model, d_model)

  #our input data q,k,v shape is (B, S ,d_model)
  def forward(self, q, k, v, mask = None):
    batch_size = q.size(0) # batchsize = B
    #1 linear projection #will output same dimension but weights will be updated by backpropagation
    #input: (B, S ,d_modl) -> output: (B, S ,d_model)
    Q = self.q_linear(q)
    K = self.k_linear(k)
    V = self.v_linear(v)

    #2.Head division and dimension transformation
    #input: (B, S ,d_model) ->  (B, S, num_heads, head_dim)->output: (B, num_heads, S, head_dim)

    Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1,2)
    K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1,2)
    V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1,2)

    #3. Scaled dot product attention
    #shape (scores) = (B, num_heads, S ,S)
    #scores = qkT / sqrt(head_dim)
    scores = (torch.matmul(Q, K.transpose(-2, -1)) // math.sqrt(self.head_dim))

    #masking to block future data
    if mask is not None:
      scores = scores.masked_fill(mask == 0, -1e9)
    attention_weights = torch.softmax(scores, dim = -1)
    #context vector =  (B, num_heads, S, head_dim)
    context = torch.matmul(attention_weights, V)


    #4. concat heads and change back to original dimension
    #output : (B, S, d_model)
    concat_context = context.transpose(1, 2).view(batch_size, -1, self.d_model)
    output = self.out_linear(concat_context)
    return output



